# ARC AGI 3 Candidate Ranker Training

This notebook trains a guarded search helper, not a direct gameplay policy.

The previous behavior cloning setup optimized action and coordinate imitation. That produced misleading action accuracy while the held out game rollout score stayed near zero. This version instead:

1. Reads source search or collector trajectory `.jsonl.gz` files.
2. Converts each transition into candidate ranking examples with explicit visual features.
3. Trains a small ranker to reorder candidates for search, without deleting the original search candidates.
4. Selects checkpoints by held out game ranking metrics, not by raw action imitation accuracy.

Use this checkpoint only as an additive prior for the stable source solver. It should not replace cached source solutions, source verified clicks, or the original 0.31 search path.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime

PROJECT_SOURCE_MODE = 'drive_dir'  # Default and recommended. Use 'drive_zip' only if you uploaded a project bundle zip.
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/ARC Prize 2026 - ARC-AGI-3')
DRIVE_PROJECT_ZIP = Path('/content/drive/MyDrive/ARC Prize 2026 - ARC-AGI-3/Local_Output/Colab_Bundles/arc_agi3_colab_bundle.zip')
DRIVE_INPUT_DATA_BASE = Path('/content/drive/MyDrive/ARC2026_AGI_3/Input_Data')
DRIVE_OUTPUT_BASE = Path('/content/drive/MyDrive/ARC2026_AGI_3/Training_Output')
LOCAL_WORKDIR = Path('/content/ARC Prize 2026 - ARC-AGI-3')
LOCAL_INPUT_DATA_BASE = Path('/content/ARC2026_AGI_3_Input_Data')
RUN_TS = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
OUTPUT_ROOT = DRIVE_OUTPUT_BASE / RUN_TS
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_INPUT_DATA_BASE.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

print('Project source mode:', PROJECT_SOURCE_MODE)
print('Drive project root:', DRIVE_PROJECT_ROOT)
print('Drive project zip:', DRIVE_PROJECT_ZIP)
print('Drive input data base:', DRIVE_INPUT_DATA_BASE)
print('Drive output base:', DRIVE_OUTPUT_BASE)
print('Local workdir:', LOCAL_WORKDIR)
print('Local input data base:', LOCAL_INPUT_DATA_BASE)
print('Output root:', OUTPUT_ROOT)


In [ ]:
import shutil

if LOCAL_WORKDIR.exists():
    shutil.rmtree(LOCAL_WORKDIR)
LOCAL_WORKDIR.parent.mkdir(parents=True, exist_ok=True)

if PROJECT_SOURCE_MODE == 'drive_dir':
    get_ipython().system('rsync -a --delete --exclude .git --exclude .venv --exclude __pycache__ --exclude .DS_Store --exclude Local_Output --exclude Gif_Demo --exclude OpenLab_Backup_* "{}"/ "{}"/'.format(DRIVE_PROJECT_ROOT, LOCAL_WORKDIR))
elif PROJECT_SOURCE_MODE == 'drive_zip':
    if not DRIVE_PROJECT_ZIP.exists():
        raise FileNotFoundError(f'Project zip not found: {DRIVE_PROJECT_ZIP}')
    get_ipython().system('unzip -q "{}" -d /content'.format(DRIVE_PROJECT_ZIP))
else:
    raise ValueError(f'Unsupported PROJECT_SOURCE_MODE: {PROJECT_SOURCE_MODE}')

get_ipython().run_line_magic('cd', str(LOCAL_WORKDIR))


In [ ]:
import importlib.util, subprocess
from pathlib import Path

def run(cmd):
    print('>>>', cmd)
    subprocess.check_call(cmd, shell=True)

def run_streaming(cmd, log_path=None):
    print('>>>', cmd)
    handle = None
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        handle = log_path.open('a', encoding='utf-8')
    try:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            if handle is not None:
                handle.write(line)
                handle.flush()
        return_code = proc.wait()
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, cmd)
    finally:
        if handle is not None:
            handle.close()

def has_module(name):
    return importlib.util.find_spec(name) is not None

run('python -m pip install -U pip wheel setuptools')

if has_module('torch'):
    print('torch already available, skipping torch/vision/audio install')
else:
    run('python -m pip install -U torch torchvision torchaudio')

if has_module('arc_agi') and has_module('arcengine'):
    print('arc_agi and arcengine already available, skipping install')
else:
    try:
        run('python -m pip install -U arc-agi==0.9.8 arcengine==0.9.3')
    except Exception:
        print('PyPI install failed, trying local wheels...')
        run('python -m pip install arc_agi_3_wheels/*.whl')

run('python - <<\'PY\'\nimport torch\nprint("torch", torch.__version__)\nprint("cuda", torch.cuda.is_available())\nif torch.cuda.is_available():\n    print("device", torch.cuda.get_device_name(0))\nPY')


In [ ]:
# Configure ranker data generation.
# Use search or collect trajectories, not raw human demonstrations, whenever possible.
# Good inputs are produced by `src.collect_openlab` as `all_episodes.jsonl.gz`, or by older collectors as `episodes.jsonl.gz`.

RUN_TAG = 'ranker_from_source_search'
OUTPUT_ROOT = DRIVE_OUTPUT_BASE / RUN_TAG / RUN_TS
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Replace this with your OpenLab aggregate path after collection.
TRAIN_EPISODE_PATHS = [
    DRIVE_INPUT_DATA_BASE / 'openlab_ranker_collect' / 'all_episodes.jsonl.gz',
]

# Keep this false by default. Raw human demos make a weak imitation dataset for hidden games.
ALLOW_HUMAN_DEMO_FALLBACK = False
HUMAN_DEMO_GZ = DRIVE_INPUT_DATA_BASE / 'arc_agi_3_public_demo_human_testing.gz'

USE_LOCAL_DATA_CACHE = True
RANKER_COORD_BUDGET = 32
RANKER_MAX_STEPS = 240
RANKER_MIN_POSITIVE_UTILITY = 0.05
RANKER_MAX_EXAMPLES = 0  # 0 means no cap.
RANKER_PROGRESS_EVERY = 1000

if ALLOW_HUMAN_DEMO_FALLBACK and not any(path.exists() for path in TRAIN_EPISODE_PATHS):
    TRAIN_EPISODE_PATHS = [HUMAN_DEMO_GZ]

TRAIN_EPISODE_PATHS = [Path(path) for path in TRAIN_EPISODE_PATHS if Path(path).exists()]
if not TRAIN_EPISODE_PATHS:
    raise FileNotFoundError(
        'No source/search trajectory gzip found. Run src.collect_openlab first and copy all_episodes.jsonl.gz into Drive input data.'
    )

if USE_LOCAL_DATA_CACHE:
    LOCAL_INPUT_DATA_BASE.mkdir(parents=True, exist_ok=True)
    local_paths = []
    for src_path in TRAIN_EPISODE_PATHS:
        local_path = LOCAL_INPUT_DATA_BASE / src_path.name
        if (not local_path.exists()) or (local_path.stat().st_size != src_path.stat().st_size):
            print(f'Copying trajectory data to local disk: {src_path} -> {local_path}')
            shutil.copy2(src_path, local_path)
        else:
            print(f'Using existing local cached data: {local_path}')
        local_paths.append(local_path)
    TRAIN_EPISODE_PATHS = local_paths

TRAIN_EPISODE_ARG = ','.join(str(path) for path in TRAIN_EPISODE_PATHS)
RANKER_DATA_PATH = OUTPUT_ROOT / 'ranker_examples.jsonl.gz'
RANKER_METADATA_PATH = OUTPUT_ROOT / 'ranker_metadata.json'

print('Output root:', OUTPUT_ROOT)
print('Trajectory paths:')
for path in TRAIN_EPISODE_PATHS:
    print(' -', path)
print('Ranker data path:', RANKER_DATA_PATH)


In [ ]:
# Build candidate ranking examples from trajectory logs.
# The builder records rare color, component shape, local contrast, recent frame change, source verified click evidence, and progress utility.

max_examples_arg = '' if RANKER_MAX_EXAMPLES == 0 else f' --max-examples {RANKER_MAX_EXAMPLES}'

build_ranker_cmd = (
    f'PYTHONUNBUFFERED=1 python -m src.build_ranker_dataset '
    f'--episodes "{TRAIN_EPISODE_ARG}" '
    f'--output "{RANKER_DATA_PATH}" '
    f'--metadata-output "{RANKER_METADATA_PATH}" '
    f'--coord-budget {RANKER_COORD_BUDGET} '
    f'--max-steps {RANKER_MAX_STEPS} '
    f'--min-positive-utility {RANKER_MIN_POSITIVE_UTILITY} '
    f'--progress-every {RANKER_PROGRESS_EVERY}'
    f'{max_examples_arg}'
)

run_streaming(build_ranker_cmd, OUTPUT_ROOT / 'build_ranker_dataset_stdout.log')


In [ ]:
# Train the guarded candidate ranker.
# The model is intentionally small because it is only a candidate ordering prior.
# It should be enabled downstream only when top probability and top margin are high.

RANKER_EPOCHS = 20
RANKER_BATCH_SIZE = 64
RANKER_HIDDEN_DIM = 128
RANKER_DROPOUT = 0.10
RANKER_LR = 3e-4
RANKER_WEIGHT_DECAY = 1e-3
RANKER_CONFIDENCE_THRESHOLD = 0.55
RANKER_MARGIN_THRESHOLD = 0.15

train_ranker_cmd = (
    f'PYTHONUNBUFFERED=1 python -m src.train_ranker '
    f'--data "{RANKER_DATA_PATH}" '
    f'--output-dir "{OUTPUT_ROOT}" '
    f'--split-mode game '
    f'--epochs {RANKER_EPOCHS} '
    f'--batch-size {RANKER_BATCH_SIZE} '
    f'--hidden-dim {RANKER_HIDDEN_DIM} '
    f'--dropout {RANKER_DROPOUT} '
    f'--lr {RANKER_LR} '
    f'--weight-decay {RANKER_WEIGHT_DECAY} '
    f'--min-target {RANKER_MIN_POSITIVE_UTILITY} '
    f'--max-candidates 48 '
    f'--confidence-threshold {RANKER_CONFIDENCE_THRESHOLD} '
    f'--margin-threshold {RANKER_MARGIN_THRESHOLD}'
)

run_streaming(train_ranker_cmd, OUTPUT_ROOT / 'train_ranker_stdout.log')


In [ ]:
# Inspect ranker metrics.
# Do not treat a high training score as proof of hidden-game generalization. The key metrics are held out game top1, MRR, confident rate, and confident top1.

import json
import pandas as pd

metrics_path = OUTPUT_ROOT / 'metrics.csv'
summary_path = OUTPUT_ROOT / 'summary.json'
metadata_path = RANKER_METADATA_PATH

display(pd.read_csv(metrics_path).tail())
print('Best ranker checkpoint:', OUTPUT_ROOT / 'checkpoints' / 'best_ranker.pth')
print('Last ranker checkpoint:', OUTPUT_ROOT / 'checkpoints' / 'last_ranker.pth')
print('Summary:', summary_path)
print('Metadata:', metadata_path)

print('
Summary JSON:')
print(json.dumps(json.loads(summary_path.read_text()), indent=2)[:4000])
print('
Dataset metadata:')
print(json.dumps(json.loads(metadata_path.read_text()), indent=2)[:4000])
